In [1]:
from pyspark.sql import SparkSession
from pyspark.ml import Pipeline, PipelineModel
from pyspark.ml.feature import StringIndexer, OneHotEncoder, VectorAssembler
from pyspark.ml.classification import RandomForestClassifier
from pyspark.ml.evaluation import MulticlassClassificationEvaluator, BinaryClassificationEvaluator
from pyspark.ml.functions import vector_to_array
from pyspark.sql import functions as F
from pymongo import MongoClient
import pandas as pd
import random, math

# Lokasi penyimpanan (ubah di sini saja bila HDFS tidak dipakai, mis. "file:///home/zhio/model_pembatalan_tersimpan")
PATH_MODEL = "model_pembatalan_tersimpan"
PATH_LAPORAN = "laporan_risiko_pembatalan"

spark = SparkSession.builder.appName("Tugas13-Sesi1").master("local[*]").getOrCreate()
spark.sparkContext.setLogLevel("ERROR")
spark.conf.set("spark.sql.shuffle.partitions", "4")
print("Sesi 1 siap. Spark:", spark.version, "| applicationId:", spark.sparkContext.applicationId)

Using Spark's default log4j profile: org/apache/spark/log4j2-defaults.properties
26/11/26 07:52:05 WARN Utils: Your hostname, vmi3356457, resolves to a loopback address: 127.0.1.1; using 13.140.149.59 instead (on interface eth0)
26/11/26 07:52:05 WARN Utils: Set SPARK_LOCAL_IP if you need to bind to another address
Using Spark's default log4j profile: org/apache/spark/log4j2-defaults.properties
Setting default log level to "WARN".
To adjust logging level use sc.setLogLevel(newLevel). For SparkR, use setLogLevel(newLevel).
26/11/26 07:52:11 WARN NativeCodeLoader: Unable to load native-hadoop library for your platform... using builtin-java classes where applicable
26/11/26 07:52:16 WARN Utils: Service 'SparkUI' could not bind on port 4040. Attempting port 4041.
26/11/26 07:52:16 WARN Utils: Service 'SparkUI' could not bind on port 4041. Attempting port 4042.
26/11/26 07:52:16 WARN Utils: Service 'SparkUI' could not bind on port 4042. Attempting port 4043.


Sesi 1 siap. Spark: 4.2.1 | applicationId: local-1795675937446


## A. Menyisipkan Data ke MongoDB
Label `dibatalkan` dibangkitkan secara probabilistik: peluang batal naik mengikuti `estimasi_pengiriman_hari` (kurva sigmoid berpusat di 7 hari), ditambah sedikit tambahan risiko untuk metode **COD**. Data dibuat dengan modul `random` bawaan Python agar bisa diserialisasi PyMongo.

In [2]:
client = MongoClient("mongodb://localhost:27017/")
db = client["ecommerce_p13"]
koleksi = db["transaksi_mentah"]
koleksi.delete_many({})  # bersihkan agar percobaan konsisten

random.seed(99)
N = 1000  # syarat minimal 800
kategori_list = ["Elektronik", "Fashion", "Kecantikan", "Rumah Tangga", "Olahraga"]
kota_list = ["Jakarta", "Bandung", "Semarang", "Surabaya", "Yogyakarta"]
metode_list = ["Transfer Bank", "COD", "E-Wallet", "Kartu Kredit"]
harga_list = [25000, 50000, 100000, 250000, 500000]

def buat_dokumen():
    est = random.randint(1, 14)
    metode = random.choice(metode_list)
    logit = (est - 7) / 2 + (0.8 if metode == "COD" else 0.0)
    p_batal = 1 / (1 + math.exp(-logit))
    return {
        "kategori": random.choice(kategori_list),
        "kota": random.choice(kota_list),
        "unit_terjual": random.randint(1, 10),
        "harga_satuan": random.choice(harga_list),
        "metode_pembayaran": metode,
        "estimasi_pengiriman_hari": est,
        "dibatalkan": 1 if random.random() < p_batal else 0,
    }

koleksi.insert_many([buat_dokumen() for _ in range(N)])
print(f"{koleksi.count_documents({})} dokumen berhasil disisipkan ke ecommerce_p13.transaksi_mentah")
print("Contoh dokumen:", koleksi.find_one({}, {"_id": 0}))

1000 dokumen berhasil disisipkan ke ecommerce_p13.transaksi_mentah
Contoh dokumen: {'kategori': 'Fashion', 'kota': 'Yogyakarta', 'unit_terjual': 3, 'harga_satuan': 50000, 'metode_pembayaran': 'Kartu Kredit', 'estimasi_pengiriman_hari': 7, 'dibatalkan': 1}


## B. Ekstraksi ke Spark DataFrame

In [3]:
df_pandas = pd.DataFrame(list(koleksi.find({}, {"_id": 0})))
df_mongo = spark.createDataFrame(df_pandas)

df_mongo.printSchema()
print("Jumlah baris:", df_mongo.count())
df_mongo.show(5)

/home/zhio/jupyter-env/lib/python3.12/site-packages/pyspark/sql/pandas/conversion.py:687: UserWarning: createDataFrame attempted Arrow optimization because 'spark.sql.execution.arrow.pyspark.enabled' is set to true; however, failed by the reason below:
  [PACKAGE_NOT_INSTALLED] PyArrow >= 18.0.0 must be installed; however, it was not found.
Attempting non-optimization as 'spark.sql.execution.arrow.pyspark.fallback.enabled' is set to true.
  warn(msg)


root
 |-- kategori: string (nullable = true)
 |-- kota: string (nullable = true)
 |-- unit_terjual: long (nullable = true)
 |-- harga_satuan: long (nullable = true)
 |-- metode_pembayaran: string (nullable = true)
 |-- estimasi_pengiriman_hari: long (nullable = true)
 |-- dibatalkan: long (nullable = true)



Jumlah baris: 1000


+------------+----------+------------+------------+-----------------+------------------------+----------+
|    kategori|      kota|unit_terjual|harga_satuan|metode_pembayaran|estimasi_pengiriman_hari|dibatalkan|
+------------+----------+------------+------------+-----------------+------------------------+----------+
|     Fashion|Yogyakarta|           3|       50000|     Kartu Kredit|                       7|         1|
|  Kecantikan|  Surabaya|           9|      500000|    Transfer Bank|                      13|         1|
|     Fashion|  Surabaya|          10|      500000|     Kartu Kredit|                      10|         1|
|Rumah Tangga|   Bandung|           3|      250000|         E-Wallet|                      13|         1|
|  Elektronik|Yogyakarta|           2|      250000|         E-Wallet|                      14|         1|
+------------+----------+------------+------------+-----------------+------------------------+----------+
only showing top 5 rows


## C. Melatih Model & Menyimpannya
Split 80:20 (`seed=99`), pipeline `StringIndexer → OneHotEncoder → VectorAssembler → RandomForestClassifier`. `handleInvalid="keep"` dipakai agar kategori yang belum pernah terlihat saat pelatihan tidak membuat model error pada data baru.

In [4]:
kolom_kategorikal = ["kategori", "kota", "metode_pembayaran"]
kolom_numerik = ["unit_terjual", "harga_satuan", "estimasi_pengiriman_hari"]

indexers = [StringIndexer(inputCol=c, outputCol=f"{c}_idx", handleInvalid="keep") for c in kolom_kategorikal]
ohe = OneHotEncoder(inputCols=[f"{c}_idx" for c in kolom_kategorikal],
                    outputCols=[f"{c}_vec" for c in kolom_kategorikal],
                    handleInvalid="keep")
assembler = VectorAssembler(inputCols=[f"{c}_vec" for c in kolom_kategorikal] + kolom_numerik,
                            outputCol="features")
rf = RandomForestClassifier(featuresCol="features", labelCol="dibatalkan",
                            numTrees=50, maxDepth=5, seed=99)

pipeline = Pipeline(stages=indexers + [ohe, assembler, rf])

train, test = df_mongo.randomSplit([0.8, 0.2], seed=99)
print("Baris latih:", train.count(), "| Baris uji:", test.count())

model = pipeline.fit(train)
pred_uji = model.transform(test)

acc = MulticlassClassificationEvaluator(labelCol="dibatalkan", predictionCol="prediction", metricName="accuracy").evaluate(pred_uji)
auc = BinaryClassificationEvaluator(labelCol="dibatalkan", rawPredictionCol="rawPrediction", metricName="areaUnderROC").evaluate(pred_uji)

# Pembanding: akurasi jika model hanya menebak kelas mayoritas
proporsi_batal = test.agg(F.avg("dibatalkan")).first()[0]
baseline = max(proporsi_batal, 1 - proporsi_batal)

print(f"Accuracy          : {acc:.4f}")
print(f"AUC-ROC           : {auc:.4f}")
print(f"Baseline mayoritas: {baseline:.4f}  (model harus jelas lebih baik dari angka ini)")

model.write().overwrite().save(PATH_MODEL)
print(f"Model disimpan ke '{PATH_MODEL}'")

Baris latih: 779 | Baris uji: 221


Accuracy          : 0.8009
AUC-ROC           : 0.8594
Baseline mayoritas: 0.5611  (model harus jelas lebih baik dari angka ini)


[Stage 77:>                                                         (0 + 1) / 1]

Model disimpan ke 'model_pembatalan_tersimpan'


**Analisis pola pembatalan pada data (disimpan dalam pandas agar tetap bisa dipakai setelah SparkSession ditutup di bagian D).** Angka ini dipakai sebagai dasar kesimpulan di bagian F.

In [5]:
analisis = df_pandas.copy()
analisis["kelompok_estimasi"] = pd.cut(analisis["estimasi_pengiriman_hari"],
                                       bins=[0, 4, 9, 14], labels=["1-4 hari", "5-9 hari", "10-14 hari"])

tabel_estimasi = (analisis.groupby("kelompok_estimasi", observed=True)["dibatalkan"]
                  .agg(jumlah="count", persen_batal=lambda s: round(s.mean() * 100, 1)))
tabel_metode = (analisis.groupby("metode_pembayaran")["dibatalkan"]
                .agg(jumlah="count", persen_batal=lambda s: round(s.mean() * 100, 1)))

print("Tingkat pembatalan per kelompok estimasi pengiriman:")
print(tabel_estimasi, "\n")
print("Tingkat pembatalan per metode pembayaran:")
print(tabel_metode)

Tingkat pembatalan per kelompok estimasi pengiriman:
                   jumlah  persen_batal
kelompok_estimasi                      
1-4 hari              265          13.2
5-9 hari              379          55.7
10-14 hari            356          95.2 

Tingkat pembatalan per metode pembayaran:
                   jumlah  persen_batal
metode_pembayaran                      
COD                   235          67.2
E-Wallet              250          51.2
Kartu Kredit          253          60.1
Transfer Bank         262          56.1


## D. Simulasi "Minggu Berikutnya"
Langkah: (1) simpan ID sesi lama, (2) `spark.stop()`, (3) buktikan sesi lama tidak bisa dipakai, (4) hapus variabel model & pipeline lama dari memori, (5) buat SparkSession baru, (6) `PipelineModel.load()` **tanpa `fit()`**.

In [6]:
id_lama = spark.sparkContext.applicationId
nama_lama = spark.sparkContext.appName

spark.stop()

# Bukti 1: sesi lama benar-benar mati
try:
    spark.range(1).count()
    print("PERINGATAN: sesi lama masih bisa dipakai (seharusnya tidak)")
except Exception as e:
    print("Bukti sesi lama sudah ditutup -> error:", type(e).__name__)

# Bukti 2: objek hasil pelatihan dibuang dari memori; satu-satunya sumber model adalah folder tersimpan
del model, pipeline, rf, pred_uji, train, test, df_mongo
print("Variabel model/pipeline hasil pelatihan sudah dihapus dari memori Python.")

Bukti sesi lama sudah ditutup -> error: AttributeError
Variabel model/pipeline hasil pelatihan sudah dihapus dari memori Python.


In [7]:
spark = SparkSession.builder.appName("Tugas13-SesiBaru").master("local[*]").getOrCreate()
spark.sparkContext.setLogLevel("ERROR")

id_baru = spark.sparkContext.applicationId
nama_baru = spark.sparkContext.appName

print(f"Sesi lama : {nama_lama} | {id_lama}")
print(f"Sesi baru : {nama_baru} | {id_baru}")
print("ID aplikasi berbeda (sesi benar-benar baru):", id_lama != id_baru)

# Memuat model TANPA fit()
model_dimuat = PipelineModel.load(PATH_MODEL)
print("\nModel dimuat dari folder tersimpan. Jumlah tahap pipeline:", len(model_dimuat.stages))
print("Tahap:", [type(s).__name__ for s in model_dimuat.stages])

Sesi lama : Tugas13-Sesi1 | local-1795675937446
Sesi baru : Tugas13-SesiBaru | local-1795676308239
ID aplikasi berbeda (sesi benar-benar baru): True



Model dimuat dari folder tersimpan. Jumlah tahap pipeline: 6
Tahap: ['StringIndexerModel', 'StringIndexerModel', 'StringIndexerModel', 'OneHotEncoderModel', 'VectorAssembler', 'RandomForestClassificationModel']


### D (lanjutan): 10 transaksi baru → disisipkan ke MongoDB (`transaksi_baru`) → diekstraksi → diprediksi

In [8]:
koleksi_baru = db["transaksi_baru"]
koleksi_baru.delete_many({})

random.seed(2024)
estimasi_baru = [1, 2, 3, 5, 7, 8, 10, 12, 13, 14]  # sengaja bervariasi untuk melihat pola prediksi
transaksi_baru = [{
    "kategori": random.choice(kategori_list),
    "kota": random.choice(kota_list),
    "unit_terjual": random.randint(1, 10),
    "harga_satuan": random.choice(harga_list),
    "metode_pembayaran": random.choice(metode_list),
    "estimasi_pengiriman_hari": e,
} for e in estimasi_baru]
koleksi_baru.insert_many(transaksi_baru)

df_baru = spark.createDataFrame(pd.DataFrame(list(koleksi_baru.find({}, {"_id": 0}))))
print("Transaksi baru diekstraksi:", df_baru.count(), "baris")

hasil = (model_dimuat.transform(df_baru)
         .withColumn("prob_batal", F.round(vector_to_array("probability")[1], 4))
         .withColumnRenamed("prediction", "prediksi_batal"))

hasil.select("kategori", "kota", "metode_pembayaran", "unit_terjual", "harga_satuan",
             "estimasi_pengiriman_hari", "prediksi_batal", "prob_batal") \
     .orderBy("estimasi_pengiriman_hari").show(truncate=False)

/home/zhio/jupyter-env/lib/python3.12/site-packages/pyspark/sql/pandas/conversion.py:687: UserWarning: createDataFrame attempted Arrow optimization because 'spark.sql.execution.arrow.pyspark.enabled' is set to true; however, failed by the reason below:
  [PACKAGE_NOT_INSTALLED] PyArrow >= 18.0.0 must be installed; however, it was not found.
Attempting non-optimization as 'spark.sql.execution.arrow.pyspark.fallback.enabled' is set to true.
  warn(msg)
                                                                                

Transaksi baru diekstraksi: 10 baris


[Stage 28:>                                                         (0 + 4) / 4]

+------------+----------+-----------------+------------+------------+------------------------+--------------+----------+
|kategori    |kota      |metode_pembayaran|unit_terjual|harga_satuan|estimasi_pengiriman_hari|prediksi_batal|prob_batal|
+------------+----------+-----------------+------------+------------+------------------------+--------------+----------+
|Rumah Tangga|Bandung   |COD              |10          |100000      |1                       |0.0           |0.2779    |
|Rumah Tangga|Semarang  |Kartu Kredit     |9           |50000       |2                       |0.0           |0.3053    |
|Kecantikan  |Surabaya  |COD              |9           |500000      |3                       |0.0           |0.3809    |
|Kecantikan  |Yogyakarta|Transfer Bank    |6           |500000      |5                       |0.0           |0.3392    |
|Fashion     |Surabaya  |COD              |3           |500000      |7                       |1.0           |0.7271    |
|Rumah Tangga|Jakarta   |Kartu K

## E. Menyimpan Laporan Akhir (Parquet, dipartisi per `kategori`)
Kolom vektor internal (`features`, `rawPrediction`, kolom hasil indexer/encoder) dibuang agar laporan bersih dan mudah dibaca alat lain.

In [9]:
laporan = hasil.select("kategori", "kota", "metode_pembayaran", "unit_terjual", "harga_satuan",
                       "estimasi_pengiriman_hari", "prediksi_batal", "prob_batal")

laporan.write.mode("overwrite").partitionBy("kategori").parquet(PATH_LAPORAN)
print(f"Laporan disimpan ke '{PATH_LAPORAN}' (partisi: kategori)")

# Verifikasi: baca kembali
cek = spark.read.parquet(PATH_LAPORAN)
print("Jumlah baris terbaca kembali:", cek.count(), "(harus sama dengan", laporan.count(), ")")
cek.orderBy("kategori", "estimasi_pengiriman_hari").show(truncate=False)

Laporan disimpan ke 'laporan_risiko_pembatalan' (partisi: kategori)


Jumlah baris terbaca kembali: 10 (harus sama dengan 10 )


+----------+-----------------+------------+------------+------------------------+--------------+----------+------------+
|kota      |metode_pembayaran|unit_terjual|harga_satuan|estimasi_pengiriman_hari|prediksi_batal|prob_batal|kategori    |
+----------+-----------------+------------+------------+------------------------+--------------+----------+------------+
|Bandung   |E-Wallet         |6           |250000      |10                      |1.0           |0.8211    |Elektronik  |
|Surabaya  |COD              |3           |500000      |7                       |1.0           |0.7271    |Fashion     |
|Jakarta   |E-Wallet         |4           |25000       |14                      |1.0           |0.8127    |Fashion     |
|Surabaya  |COD              |9           |500000      |3                       |0.0           |0.3809    |Kecantikan  |
|Yogyakarta|Transfer Bank    |6           |500000      |5                       |0.0           |0.3392    |Kecantikan  |
|Bandung   |Kartu Kredit     |6 

## F. Kesimpulan

> **Petunjuk pengisian:** angka bertanda «...» harus diganti dengan nilai dari output sel di atas (accuracy, AUC-ROC, tabel analisis). Hapus petunjuk ini setelah selesai.

**Alur kerja end-to-end.** Sebanyak 1000 transaksi dummy dibangkitkan dengan modul `random` dan disisipkan ke MongoDB (`ecommerce_p13.transaksi_mentah`), lalu diekstraksi menjadi Spark DataFrame. Pipeline `StringIndexer → OneHotEncoder → VectorAssembler → RandomForestClassifier` dilatih dengan split 80:20 (`seed=99`) dan menghasilkan accuracy «...» serta AUC-ROC «...», dibandingkan dengan baseline tebakan kelas mayoritas «...». Model lalu disimpan ke `model_pembatalan_tersimpan`. Untuk membuktikan model persistence, `SparkSession` ditutup, variabel model dihapus, dan sesi baru dibuat (ID aplikasi berbeda); model dimuat dengan `PipelineModel.load()` tanpa `fit()` ulang dan dipakai memprediksi 10 transaksi baru dari koleksi `transaksi_baru`. Hasilnya disimpan sebagai Parquet terpartisi per `kategori` dan diverifikasi dengan membaca ulang (`count()` sesuai).

**Temuan.** Pada data, tingkat pembatalan naik dari «...%» pada estimasi 1-4 hari menjadi «...%» pada estimasi 10-14 hari, dan model memberi probabilitas batal yang meningkat searah dengan estimasi pengiriman pada 10 transaksi baru.

**Rekomendasi bisnis.** Tim operasional sebaiknya menjadikan estimasi pengiriman sebagai pemicu intervensi: transaksi dengan estimasi ≥ «ambang dari tabel, mis. 8» hari ditawari opsi pengiriman lebih cepat, diberi notifikasi proaktif status pengiriman, dan diprioritaskan saat pengepakan. Setelah diterapkan, ukur apakah tingkat pembatalan kelompok tersebut turun dibanding sebelum intervensi.

**Keterbatasan.** Data ini simulasi dan label sengaja dibuat bergantung pada estimasi pengiriman, sehingga akurasi dan AUC mencerminkan aturan yang kita tanam sendiri, bukan perilaku pelanggan sebenarnya. Sebelum dipakai di produksi, model harus dilatih ulang dan diuji pada data transaksi nyata.

In [11]:
client.close()
spark.stop()
print("Koneksi MongoDB & SparkSession ditutup.")

Koneksi MongoDB & SparkSession ditutup.
